# The first step we install sentence-transformers using pip

In [1]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# then we reuse the same path pattern as our rule-based script
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Reuse the same path pattern as your rule-based script
BASE_DIR = os.getcwd()
DATA_DIR = os.path.normpath(os.path.join(BASE_DIR, "..", "data"))
CLEAN_JOBS_PATH = os.path.join(DATA_DIR, "clean_jobs.csv")
TAXONOMY_PATH = os.path.join(DATA_DIR, "skill_taxonomy.csv")
OUTPUT_PATH = os.path.join(DATA_DIR, "semantic_skills.csv")

# Load a small, fast model — good enough for skill matching
model = SentenceTransformer("all-MiniLM-L6-v2")

# Load taxonomy and build embeddings for each canonical skill
tax = pd.read_csv(TAXONOMY_PATH)
canonical_skills = tax["skill"].tolist()
skill_embeddings = model.encode(canonical_skills, convert_to_tensor=True)

df = pd.read_csv(CLEAN_JOBS_PATH)

def semantic_match(text, threshold=0.55, top_k=5):
    # Split description into candidate phrases (simple approach: sentences or n-grams)
    chunks = [c.strip() for c in str(text).split(".") if c.strip()]
    if not chunks:
        return []

    chunk_embeddings = model.encode(chunks, convert_to_tensor=True)
    matches = set()

    for i, chunk_emb in enumerate(chunk_embeddings):
        sims = util.cos_sim(chunk_emb, skill_embeddings)[0]
        top_results = sims.topk(min(top_k, len(canonical_skills)))
        for score, idx in zip(top_results.values, top_results.indices):
            if score.item() >= threshold:
                matches.add(canonical_skills[idx.item()])

    return sorted(matches)

df["semantic_skills"] = df["clean_description"].apply(semantic_match)
df["semantic_skill_count"] = df["semantic_skills"].apply(len)

df[["job_id", "job_title", "semantic_skills", "semantic_skill_count"]].to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 932.06it/s]


Saved to c:\Users\Admin\Desktop\Internship Task\Job Skill extraction\Job_Skill_Extraction\data\semantic_skills.csv


In [3]:
import pandas as pd
tax = pd.read_csv("../data/skill_taxonomy.csv")
print(tax.columns.tolist())
print(tax.head(10))

['skill', 'category', 'aliases']
        skill     category                    aliases
0      Python  Programming        python3;python 3;py
1        Java  Programming                    java se
2  JavaScript  Programming          js;javascript es6
3        HTML  Programming                      html5
4         CSS  Programming                       css3
5       React  Programming           react.js;reactjs
6        Node  Programming             node.js;nodejs
7      Django  Programming           django framework
8       Flask  Programming            flask framework
9         SQL     Database  structured query language
